# EmotiWrite

Notebook para clasificación de emociones en texto.

## Objetivo
Construir un flujo de trabajo completo para preparar datos


## 1. Preparar datos


In [12]:
# Configuracion inicial para esta etapa
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)

print(f"Semilla fija: {SEED}")

Semilla fija: 42


Intenta leer archivos CSV o JSON. Si no encuentra archivos, crea un dataset de ejemplo para que el notebook sea ejecutable desde el inicio.

In [13]:
# Cargar datos (prioriza data/, si no usa dataset generico)
DATA_PATHS = [
    Path("data/emotions.csv"),
    Path("data/emotions.json"),
    Path("emotions.csv"),
    Path("emotions.json"),
]


def load_dataset(paths):
    for path in paths:
        if path.exists() and path.suffix.lower() == ".csv":
            print(f"Dataset cargado desde CSV: {path}")
            return pd.read_csv(path)

        if path.exists() and path.suffix.lower() == ".json":
            print(f"Dataset cargado desde JSON: {path}")
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)
            return pd.DataFrame(data)

    print("No se encontro dataset en data/. Se usa dataset generico de ejemplo.")
    generic_data = {
        "text": [
            "Hoy me siento muy feliz por los resultados",
            "Estoy triste y sin energia",
            "Este retraso me da mucha rabia",
            "Que sorpresa tan increible",
            "Tengo miedo de hablar en publico",
            "Me siento en paz esta manana",
            "No soporto esta situacion",
            "Estoy emocionado por el viaje",
            "Me siento desmotivado hoy",
            "Estoy agradecido por todo",
        ],
        "emotion": [
            "joy",
            "sadness",
            "anger",
            "surprise",
            "fear",
            "calm",
            "anger",
            "joy",
            "sadness",
            "joy",
        ],
    }

    if len(generic_data["text"]) != len(generic_data["emotion"]):
        raise ValueError("El dataset generico tiene longitudes inconsistentes.")

    return pd.DataFrame(generic_data)


df = load_dataset(DATA_PATHS).copy()

# Normalizar columnas esperadas
candidate_text_cols = ["text", "texto", "sentence", "content"]
candidate_label_cols = ["emotion", "label", "sentiment", "etiqueta"]

text_col = next((c for c in candidate_text_cols if c in df.columns), None)
label_col = next((c for c in candidate_label_cols if c in df.columns), None)

if text_col is None or label_col is None:
    raise ValueError(
        f"No se encontraron columnas validas. Columnas disponibles: {list(df.columns)}"
    )

df = df[[text_col, label_col]].rename(columns={text_col: "text", label_col: "emotion"})

display(df.head())
print("\nTamano del dataset:", df.shape)
print("\nClases:")
print(df["emotion"].value_counts())
print("\nValores nulos:")
print(df.isnull().sum())

No se encontro dataset en data/. Se usa dataset generico de ejemplo.


,text,emotion
0,Hoy me siento muy feliz por los resultados,joy
1,Estoy triste y sin energia,sadness
2,Este retraso me da mucha rabia,anger
3,Que sorpresa tan increible,surprise
4,Tengo miedo de hablar en publico,fear



Tamano del dataset: (10, 2)

Clases:
emotion
joy         3
sadness     2
anger       2
surprise    1
fear        1
calm        1
Name: count, dtype: int64

Valores nulos:
text       0
emotion    0
dtype: int64


Se define una función reutilizable para estandarizar texto: minúsculas, limpieza de URLs, signos y espacios extra.

In [14]:
# Limpiar y normalizar texto
URL_RE = re.compile(r"https?://\S+|www\.\S+")
NON_ALNUM_RE = re.compile(r"[^a-zA-Z0-9áéíóúñüÁÉÍÓÚÑÜ\s]")
MULTISPACE_RE = re.compile(r"\s+")


def clean_text(text):
    text = str(text).lower()
    text = URL_RE.sub(" ", text)
    text = NON_ALNUM_RE.sub(" ", text)
    text = MULTISPACE_RE.sub(" ", text).strip()
    return text


df["text"] = df["text"].fillna("")
df["emotion"] = df["emotion"].fillna("unknown")
df["clean_text"] = df["text"].map(clean_text)

# Eliminar filas vacias tras limpieza
df = df[df["clean_text"].str.len() > 0].reset_index(drop=True)

display(df[["text", "clean_text", "emotion"]].head())
print("\nTamano luego de limpieza:", df.shape)

,text,clean_text,emotion
0,Hoy me siento muy feliz por los resultados,hoy me siento muy feliz por los resultados,joy
1,Estoy triste y sin energia,estoy triste y sin energia,sadness
2,Este retraso me da mucha rabia,este retraso me da mucha rabia,anger
3,Que sorpresa tan increible,que sorpresa tan increible,surprise
4,Tengo miedo de hablar en publico,tengo miedo de hablar en publico,fear



Tamano luego de limpieza: (10, 3)


Estratificación
Es una forma de dividir los datos manteniendo la misma proporción de clases en entrenamiento y prueba. Esto evita que el conjunto de prueba quede “desbalanceado” por azar.

In [15]:
# Dividir datos en entrenamiento y prueba (sin vectorizar aun)
X = df["clean_text"]
y = df["emotion"]

class_counts = y.value_counts()
can_stratify = (y.nunique() > 1) and (class_counts.min() >= 2)
stratify_target = y if can_stratify else None

if not can_stratify:
    print("Aviso: se desactiva stratify porque hay clases con menos de 2 ejemplos.")
    print("Clases con conteo actual:")
    print(class_counts)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=stratify_target,
)

print("\nSplit completado:")
print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print("y_train:", y_train.shape)
print("y_test: ", y_test.shape)

print("\nDistribucion en train:")
print(y_train.value_counts(normalize=True).round(3))

print("\nDistribucion en test:")
print(y_test.value_counts(normalize=True).round(3))

Aviso: se desactiva stratify porque hay clases con menos de 2 ejemplos.
Clases con conteo actual:
emotion
joy         3
sadness     2
anger       2
surprise    1
fear        1
calm        1
Name: count, dtype: int64

Split completado:
X_train: (8,)
X_test:  (2,)
y_train: (8,)
y_test:  (2,)

Distribucion en train:
emotion
joy         0.375
anger       0.250
calm        0.125
fear        0.125
surprise    0.125
Name: proportion, dtype: float64

Distribucion en test:
emotion
sadness    1.0
Name: proportion, dtype: float64
